# Alias-Free Oscillator Synchronization via Additive Synthesis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import IPython.display as ipd
import time
import hasy_extras

plt.rcParams['figure.figsize'] = (10, 4)

### Input Parameters

In [ ]:
fs_Hz = 96000
duration_s = 1

sync_mode = 'hard'  # 'hard' or 'mirrored' or 'pulsar' or 'dbg_bypass'
period_ratio = 11/8
f_lead_Hz = 94
following_waveform = 'sine'   # 'sine' or 'cosine' or 'saw' or 'square' or 'triangle' or 'pulse' or 'heart' or 'sandstorm'

N_MAX = 512 # max number of harmonics

### quality setting for parameter sweeps
# SAMPLES_PER_UPDATE = 1000   # draft mode
SAMPLES_PER_UPDATE =    5   # reproduce paper results (corresponds to HASY)
# SAMPLES_PER_UPDATE =    1   # maximum quality (per-sample updates)

### Set Following Waveform Fourier Coefficients

In [ ]:
def coefs_zero(N_max: int):
  return (0.0, np.zeros(N_max), np.zeros(N_max))

def coefs_sine(N_max: int):
  a, b = np.zeros(N_max), np.zeros(N_max)
  b[0] = 1.0
  return (0.0, a, b)

def coefs_cosine(N_max: int):
  a, b = np.zeros(N_max), np.zeros(N_max)
  a[0] = 1.0
  return (0.0, a, b)

def coefs_saw(N_max: int):
  a, b = np.zeros(N_max), np.zeros(N_max)
  n = np.arange(1, N_max+1)
  b[n-1] = (-1)**n * -2 / (np.pi*n)
  return (0.0, a, b)

def coefs_square(N_max: int):
  a, b = np.zeros(N_max), np.zeros(N_max)
  k = np.arange(1, (N_max//2)+1)
  n = 2*k-1
  b[n-1] = 4 / (np.pi * n)
  return (0.0, a, b)

def coefs_triangle(N_max: int):
  a, b = np.zeros(N_max), np.zeros(N_max)
  k = np.arange(1, (N_max//2)+1)
  n = 2*k-1
  b[n-1] = (-1)**k * -8 / (np.pi*n)**2
  return (0.0, a, b)

def coefs_pulse(N_max: int):
  a = np.ones(N_max) / N_max
  b = np.zeros(N_max)
  return (0.0, a, b)


def get_coefs_magnitude(coefs: tuple, ignore_DC: bool =True):
  a0, a, b = coefs
  mag = np.sqrt(np.square(a) + np.square(b))
  if not ignore_DC:
    mag = np.concat(([a0], mag))
  return mag


In [ ]:
match following_waveform:
  case 'sine':
    coefs_follow = coefs_sine(N_MAX)
  case 'cosine':
    coefs_follow = coefs_cosine(N_MAX)
  case 'saw':
    coefs_follow = coefs_saw(N_MAX)
  case 'square':
    coefs_follow = coefs_square(N_MAX)
  case 'triangle':
    coefs_follow = coefs_triangle(N_MAX)
  case 'pulse':
    coefs_follow = coefs_pulse(N_MAX)
  case 'heart':
    coefs_follow = hasy_extras.coefs_heart(N_MAX)
  case 'sandstorm':
    coefs_follow = hasy_extras.coefs_sandstorm(N_MAX)
  case _:
    print(f'Unknown waveform: {following_waveform}')

### Spectral Resampling

In [ ]:
def versinc(x: float):
  if np.abs(x) > 1e-12:
    return (1 - np.cos(np.pi*x)) / (np.pi*x)
  else:
    return 0.0


def prerotate(coefs_follow: tuple, tau: float):
  a0_follow, a_follow, b_follow = coefs_follow
  N_max = len(a_follow)

  a0_rot = a0_follow

  a_rot = np.zeros(N_max)
  b_rot = np.zeros(N_max)
  for n in range(1, N_max+1):
    # note that the sign is flipped compared to the paper since we are using a and b instead of Re{c} and Im{c}
    arg = 2 * np.pi * tau * n
    a_rot[n-1] = a_follow[n-1] * np.cos(arg) + b_follow[n-1] * -np.sin(arg)
    b_rot[n-1] = a_follow[n-1] * np.sin(arg) + b_follow[n-1] *  np.cos(arg)

  return (a0_rot, a_rot, b_rot)


def transform_hard(coefs_rot: tuple, period_ratio: float):
  a0_rot, a_rot, b_rot = coefs_rot
  N_max = len(a_rot)

  a0_sync = a0_rot
  for k in range(1, N_max+1):
    arg = k*period_ratio
    a0_sync += a_rot[k-1] * np.sinc(arg)

  a_sync = np.zeros(N_max)
  b_sync = np.zeros(N_max)
  for n in range(1, N_max+1):
    for k in range(1, N_max+1):
      arg_p = n + k*period_ratio
      arg_n = n - k*period_ratio
      a_sync[n-1] += a_rot[k-1] * (np.sinc(arg_n) + np.sinc(arg_p))
      b_sync[n-1] += b_rot[k-1] * (np.sinc(arg_n) - np.sinc(arg_p))

  return (a0_sync, a_sync, b_sync)


def transform_mirrored(coefs_rot: tuple, period_ratio: float):
  a0_rot, a_rot, b_rot = coefs_rot
  N_max = len(a_rot)

  a0_sync = a0_rot
  for k in range(1, N_max+1):
    arg = 2*k*period_ratio
    a0_sync += a_rot[k-1] * np.sinc(arg) - b_rot[k-1] * versinc(arg)

  a_sync = np.zeros(N_max)
  b_sync = np.zeros(N_max)
  for n in range(1, N_max+1):
    for k in range(1, N_max+1):
      arg_p = n + 2*k*period_ratio
      arg_n = n - 2*k*period_ratio
      a_sync[n-1] +=  a_rot[k-1] * (np.sinc(arg_n) + np.sinc(arg_p)) \
                    + b_rot[k-1] * (versinc(arg_n) - versinc(arg_p))

  return (a0_sync, a_sync, b_sync)


def transform_pulsar(coefs_rot: tuple, period_ratio: float):
  a0_rot, a_rot, b_rot = coefs_rot
  N_max = len(a_rot)

  a0_sync = 0

  a_sync = np.zeros(N_max)
  b_sync = np.zeros(N_max)
  for n in range(1, N_max+1):
    for k in range(1, N_max+1):
      arg_p = n/period_ratio + k
      arg_n = n/period_ratio - k
      a_sync[n-1] += a_rot[k-1]/period_ratio * (np.sinc(arg_n) + np.sinc(arg_p))
      b_sync[n-1] += b_rot[k-1]/period_ratio * (np.sinc(arg_n) - np.sinc(arg_p))

  return (a0_sync, a_sync, b_sync)


def spectral_resampling(coefs_follow: tuple, period_ratio: float, sync_mode: str):
  # unpack tuple, validate inputs
  a0_follow, a_follow, b_follow = coefs_follow
  assert len(a_follow) == len(b_follow)
  assert period_ratio > 0

  match sync_mode:
    case 'hard':
      tau = -period_ratio / 2
      coefs_rot = prerotate(coefs_follow, tau)
      return transform_hard(coefs_rot, period_ratio)
    case 'mirrored':
      tau = -period_ratio
      coefs_rot = prerotate(coefs_follow, tau)
      return transform_mirrored(coefs_rot, period_ratio)
    case 'pulsar':
      tau = -0.5
      coefs_rot = prerotate(coefs_follow, tau)
      return transform_pulsar(coefs_rot, period_ratio)
    case 'dbg_bypass':
      return coefs_follow
    case _:
      raise ValueError(f"Invalid sync mode: {sync_mode}")


In [ ]:
coefs_sync = spectral_resampling(coefs_follow, period_ratio, sync_mode)


In [ ]:
match sync_mode:
  case 'hard':
    coef_label = r'$|\bar{c}_n|$'
  case 'mirrored':
    coef_label = r'$|\hat{c}_n|$'
  case 'pulsar':
    coef_label = r'$|\ddot{c}_n|$'
  case _:
    coef_label = 'unknown sync mode'

idx = np.arange(1, N_MAX+1)
plt.semilogy(idx, get_coefs_magnitude(coefs_follow), 'o', label='$|c_n|$')
plt.semilogy(idx, get_coefs_magnitude(coefs_sync),   '.', label=coef_label)
plt.title('Fourier coefficients before and after spectral transform')
plt.xlabel('coefficient index $n$')
plt.ylabel('magnitude (log)')
# plt.xlim((-2,52))
plt.grid(True)
plt.legend()
plt.show()

### Additive Synthesis

In [ ]:
class FourierOscillator:
  def __init__(self, fs_Hz: float, N_max: int, ignore_DC: bool =True):
    self.fs_Hz = fs_Hz
    self.N_max = N_max
    self.ignore_DC = ignore_DC
    self.coefs = (0.0, np.zeros(N_max), np.zeros(N_max))
    self.f_fund_Hz = 440.0
    self.phase = 0.0

  def set_coefs(self, coefs: tuple):
    self.coefs = coefs

  def set_f_fund(self, f_fund_Hz: float):
    self.f_fund_Hz = f_fund_Hz

  def generate(self, n_smpl: int =1) -> np.ndarray:
    a0, a, b = self.coefs
    phase_inc = self.f_fund_Hz/self.fs_Hz
    phase_base = np.zeros(n_smpl)
    for i in range(n_smpl):
      phase_base[i] = self.phase
      self.phase += phase_inc
      if self.phase > 1:
        self.phase = self.phase-1


    N_lim = self.fs_Hz*0.5/self.f_fund_Hz
    N_lim = int(np.floor(np.nextafter(N_lim, 0))) # force strictly smaller
    N_lim = min(N_lim, self.N_max)

    if self.ignore_DC == True:
      x = 0.0
    else:
      x = np.ones(n_smpl) * a0

    for n in range(1, N_lim+1):
      phase = 2*np.pi * ((phase_base * n) % 1)
      x += a[n-1] * np.cos(phase)
      x += b[n-1] * np.sin(phase)

    return x


In [ ]:
add_synth = FourierOscillator(fs_Hz, N_MAX)

f_fund_Hz = f_lead_Hz
if sync_mode == 'mirrored':
  f_fund_Hz = f_lead_Hz * 0.5
add_synth.set_f_fund(f_fund_Hz)

add_synth.set_coefs(coefs_sync)

s_static = add_synth.generate(int(duration_s*fs_Hz))  # synchronized output (static parameters)


plt.plot(s_static[:int(4*fs_Hz/f_lead_Hz)])
plt.show()

ipd.Audio(s_static, rate=fs_Hz)

### Vectorized Functions
Require less patience when modulating parameters over time

In [ ]:
def versinc_vectorized(x, eps: float = 1e-12):
  x_arr = np.asarray(x)
  out = np.zeros_like(x_arr, dtype=float)

  mask = np.abs(x_arr) > eps  # use mask to avoid division by zero
  num = 1 - np.cos(np.pi * x_arr)
  den = np.pi * x_arr
  np.divide(num, den, out=out, where=mask)

  return out.item() if np.isscalar(x) else out


def prerotate_vectorized(coefs_follow: tuple, tau: float):
  a0_follow, a_follow, b_follow = coefs_follow
  N_max = len(a_follow)

  a0_rot = a0_follow

  n = np.arange(1, N_max + 1)
  arg = 2 * np.pi * tau * n
  c = np.cos(arg)
  s = np.sin(arg)
  # note to self: the sign for the sin terms is flipped compared to the paper since we are using a and b instead of Re{c} and Im{c}
  a_rot = a_follow * c - b_follow * s
  b_rot = a_follow * s + b_follow * c

  return (a0_rot, a_rot, b_rot)


def transform_hard_vectorized(coefs_rot: tuple, period_ratio: float):
  a0_rot, a_rot, b_rot = coefs_rot
  N_max = len(a_rot)

  k = np.arange(1, N_max + 1)
  a0_sync = a0_rot + np.dot(a_rot, np.sinc(k * period_ratio))

  n = k.reshape(-1, 1)          # (N, 1)
  arg_row = (k * period_ratio).reshape(1, -1)  # (1, N)

  sinc_minus = np.sinc(n - arg_row)
  sinc_plus  = np.sinc(n + arg_row)

  a_sync = (sinc_minus + sinc_plus) @ a_rot
  b_sync = (sinc_minus - sinc_plus) @ b_rot

  return (a0_sync, a_sync, b_sync)


def transform_mirrored_vectorized(coefs_rot: tuple, period_ratio: float):
  a0_rot, a_rot, b_rot = coefs_rot
  N_max = len(a_rot)

  k = np.arange(1, N_max + 1)
  arg = (2 * k * period_ratio)

  a0_sync = (
    a0_rot
    + np.dot(a_rot, np.sinc(arg))
    - np.dot(b_rot, versinc_vectorized(arg))
  )

  n = k.reshape(-1, 1)      # (N, 1)
  arg_row = arg.reshape(1, -1)   # (1, N)
  arg_minus = n - arg_row
  arg_plus  = n + arg_row

  a_sync =  (np.sinc(arg_minus) + np.sinc(arg_plus)) @ a_rot \
          + (versinc_vectorized(arg_minus) - versinc_vectorized(arg_plus)) @ b_rot
  b_sync = np.zeros(N_max)

  return (a0_sync, a_sync, b_sync)


def transform_pulsar_vectorized(coefs_rot: tuple, period_ratio: float):
  a0_rot, a_rot, b_rot = coefs_rot
  N_max = len(a_rot)

  a0_sync = 0

  k = np.arange(1, N_max + 1).reshape(1, -1)  # (1, N)
  arg_col = (k/period_ratio).reshape(-1, 1)   # (N, 1)
  sinc_minus = np.sinc(arg_col - k)
  sinc_plus  = np.sinc(arg_col + k)

  a_sync = (sinc_minus + sinc_plus) @ (a_rot/period_ratio)
  b_sync = (sinc_minus - sinc_plus) @ (b_rot/period_ratio)

  return (a0_sync, a_sync, b_sync)


def spectral_resampling_vectorized(coefs_follow: tuple, period_ratio: float, sync_mode: str):
  # unpack tuple, validate inputs
  a0_follow, a_follow, b_follow = coefs_follow
  assert len(a_follow) == len(b_follow)
  assert period_ratio > 0

  match sync_mode:
    case 'hard':
      tau = -period_ratio / 2
      coefs_rot = prerotate_vectorized(coefs_follow, tau)
      return transform_hard_vectorized(coefs_rot, period_ratio)
    case 'mirrored':
      tau = -period_ratio
      coefs_rot = prerotate_vectorized(coefs_follow, tau)
      return transform_mirrored_vectorized(coefs_rot, period_ratio)
    case 'pulsar':
      tau = -0.5
      coefs_rot = prerotate_vectorized(coefs_follow, tau)
      return transform_pulsar_vectorized(coefs_rot, period_ratio)
    case 'dbg_bypass':
      return coefs_follow
    case _:
      raise ValueError(f"Invalid sync mode: {sync_mode}")


class FourierOscillator_vectorized:
  def __init__(self, fs_Hz: float, N_max: int, ignore_DC: bool =True):
    self.fs_Hz = fs_Hz
    self.N_max = N_max
    self.ignore_DC = ignore_DC
    self.coefs = (0.0, np.zeros(N_max), np.zeros(N_max))
    self.f_fund_Hz = 440.0
    self.PHASE_PERIOD = np.uint64(2**48)
    self.phase = np.uint64(0)
    self.phase_inc = np.uint64(0)

  def set_coefs(self, coefs: tuple):
    self.coefs = coefs

  def set_f_fund(self, f_fund_Hz: float):
    self.f_fund_Hz = f_fund_Hz
    self.phase_inc = np.uint64(round(self.f_fund_Hz/self.fs_Hz * self.PHASE_PERIOD))

  def generate(self, n_smpl: int =1) -> np.ndarray:
    # limit memory usage; the bound also keeps phase_next from wrapping past 2**64,
    # which would break the np.arange() below at 2**16*fs_Hz/f_fund_Hz
    assert n_smpl <= self.fs_Hz 
    a0, a, b = self.coefs

    # limit harmonics to Nyquist frequency
    N_lim = self.fs_Hz*0.5/self.f_fund_Hz
    N_lim = int(np.floor(np.nextafter(N_lim, 0))) # force strictly smaller
    N_lim = min(N_lim, self.N_max)

    n = np.arange(1, N_lim+1, dtype=np.uint64)

    # integer phase accumulator
    modulo_bitmask = self.PHASE_PERIOD-1
    phase_next = self.phase + n_smpl*self.phase_inc
    phase_fund = np.arange(self.phase, phase_next, self.phase_inc, dtype=np.uint64)
    self.phase = np.bitwise_and(phase_next, modulo_bitmask) # increment phase for next cycle

    phases_int = np.outer(phase_fund,n)
    phases_int = np.bitwise_and(phases_int, modulo_bitmask)
    phases_rad = 2*np.pi * (phases_int/self.PHASE_PERIOD)

    if self.ignore_DC == True:
      x = np.zeros(n_smpl)
    else:
      x = np.ones(n_smpl) * a0

    x += np.dot(np.cos(phases_rad), a[:N_lim]) + np.dot(np.sin(phases_rad), b[:N_lim])

    return x


## Audio Examples

In [ ]:
# additional definitions
NOTE_E2_Hz = 82.40689

# wrapper function that moulates P over time and produces plots and audio player
def modulate_p_sim( coefs_follow: tuple,
                    sync_mode: str,
                    f_fund_Hz: float,
                    period_ratios: np.ndarray,
                    samples_per_update: int,
                    fs_Hz: float =fs_Hz,
                    plot_xlim: tuple =(0,0.1)):
  print(f'moudlate P with update-rate = {fs_Hz/samples_per_update} Hz\n')
  add_synth_vec = FourierOscillator_vectorized(fs_Hz=fs_Hz, N_max=len(coefs_follow[1]))
  add_synth_vec.set_f_fund(f_fund_Hz)

  s_sync_mod = np.zeros(len(period_ratios)*samples_per_update)
  for _i, p in enumerate(tqdm(period_ratios)):
    coefs_sync = spectral_resampling_vectorized(coefs_follow, p, sync_mode)
    add_synth_vec.set_coefs(coefs_sync)
    _j = _i * samples_per_update
    s_sync_mod[_j: _j+samples_per_update] = add_synth_vec.generate(samples_per_update)

  # plot signal excerpt
  t_sweep = np.arange(len(s_sync_mod),dtype=np.float64) / fs_Hz
  plt.plot(t_sweep, s_sync_mod, linewidth=0.5)
  plt.xlim(plot_xlim)
  plt.show()

  # plot spectrogram
  N_fft=4096
  plt.specgram(s_sync_mod,
              NFFT=N_fft, Fs=fs_Hz, scale_by_freq=False, mode='magnitude', noverlap=N_fft//2,
              cmap='Greys', vmin=-120, vmax=0)
  plt.colorbar()
  # plt.show()

  # audio player
  return ipd.Audio(s_sync_mod, rate=fs_Hz)



### Sine Hard Sync
Sine hard sync with linear rising $f_\textrm{follow}$ via sweep of period ratio $P$.

In [ ]:
sweep_duration_s = 5
num_updates = int(sweep_duration_s*fs_Hz/SAMPLES_PER_UPDATE)

modulate_p_sim( coefs_follow        = coefs_sine(N_MAX),
                sync_mode           ='hard',
                f_fund_Hz           = 100,
                period_ratios       = np.linspace(0.5, 100, num=num_updates),
                samples_per_update  = SAMPLES_PER_UPDATE )

### Sine Mirrored Sync
Sine mirrored sync with linear rising $f_\textrm{follow}$ via sweep of period ratio $P$.

In [ ]:
sweep_duration_s = 5
num_updates = int(sweep_duration_s*fs_Hz/SAMPLES_PER_UPDATE)

modulate_p_sim( coefs_follow        = coefs_sine(N_MAX),
                sync_mode           ='mirrored',
                f_fund_Hz           = 100/2,
                period_ratios       = np.linspace(0.5, 100, num=num_updates),
                samples_per_update  = SAMPLES_PER_UPDATE )

### Sine Pulsar Sync with Linear Sweep of $f_\textrm{follow}$
Linear rising $f_\textrm{follow}$ via sweep of period ratio.

In [ ]:
sweep_duration_s = 5
num_updates = int(sweep_duration_s*fs_Hz/SAMPLES_PER_UPDATE)

modulate_p_sim( coefs_follow        = coefs_sine(N_MAX),
                sync_mode           ='pulsar',
                f_fund_Hz           = 100,
                period_ratios       = np.linspace(0.5, 100, num=num_updates),
                samples_per_update  = SAMPLES_PER_UPDATE )

### Sawtooth Hard Sync
Sawtooth hard sync with exponentially decaying $f_\textrm{follow}$ via sweep of period ratio $P$.

In [ ]:
sweep_duration_s = 3
num_updates = int(sweep_duration_s*fs_Hz/SAMPLES_PER_UPDATE)

modulate_p_sim( coefs_follow        = coefs_saw(N_MAX),
                sync_mode           ='hard',
                f_fund_Hz           = NOTE_E2_Hz,
                period_ratios       = np.geomspace(20, 0.5, num=num_updates),
                samples_per_update  = SAMPLES_PER_UPDATE,
                plot_xlim           = (0,0.05) )

### Sawtooth Mirrored Sync
Sawtooth mirrored sync with exponentially decaying $f_\textrm{follow}$ via sweep of period ratio $P$.

In [ ]:
sweep_duration_s = 3
num_updates = int(sweep_duration_s*fs_Hz/SAMPLES_PER_UPDATE)

modulate_p_sim( coefs_follow        = coefs_saw(N_MAX),
                sync_mode           ='mirrored',
                f_fund_Hz           = NOTE_E2_Hz/2,
                period_ratios       = np.geomspace(20, 0.5, num=num_updates),
                samples_per_update  = SAMPLES_PER_UPDATE,
                plot_xlim           = (0,0.025) )

### Sawtooth Pulsar Sync
Sawtooth pulsar sync with exponentially decaying $f_\textrm{follow}$ via sweep of period ratio $P$.

In [ ]:
sweep_duration_s = 3
num_updates = int(sweep_duration_s*fs_Hz/SAMPLES_PER_UPDATE)

modulate_p_sim( coefs_follow        = coefs_saw(N_MAX),
                sync_mode           ='pulsar',
                f_fund_Hz           = NOTE_E2_Hz,
                period_ratios       = np.geomspace(20, 0.5, num=num_updates),
                samples_per_update  = SAMPLES_PER_UPDATE )

## Plot Examples

### Mirrored Sync applied to Sandstorm Waveform 

In [ ]:
periodratio_exmpl = 3/2
duration_exmpl_s = 1
coefs_follow_exmpl = hasy_extras.coefs_sandstorm(N_MAX)
coefs_sync_exmpl = spectral_resampling_vectorized(coefs_follow_exmpl, periodratio_exmpl, 'mirrored')
add_synth_vec = FourierOscillator_vectorized(fs_Hz=fs_Hz, N_max=N_MAX)
add_synth_vec.set_coefs(coefs_sync_exmpl)
add_synth_vec.set_f_fund(NOTE_E2_Hz/2)
s_exmpl = add_synth_vec.generate(int(duration_exmpl_s*fs_Hz))

# plot signal excerpt
t_exmpl = np.linspace(0.0, duration_exmpl_s, int(duration_exmpl_s*fs_Hz), endpoint=False)
plt.plot(t_exmpl, s_exmpl, linewidth=0.5)
plt.xlim((0,0.05))
plt.show()

ipd.Audio(s_exmpl, rate=fs_Hz)

### Pulsar Sync applied to Heart Waveform 

In [ ]:
periodratio_exmpl = 3/2
duration_exmpl_s = 1
coefs_follow_exmpl = hasy_extras.coefs_heart(N_MAX)
coefs_sync_exmpl = spectral_resampling_vectorized(coefs_follow_exmpl, periodratio_exmpl, 'pulsar')
add_synth_vec = FourierOscillator_vectorized(fs_Hz=fs_Hz, N_max=N_MAX)
add_synth_vec.set_coefs(coefs_sync_exmpl)
add_synth_vec.set_f_fund(NOTE_E2_Hz)
s_exmpl = add_synth_vec.generate(int(duration_exmpl_s*fs_Hz))

# plot signal excerpt
t_exmpl = np.linspace(0.0, duration_exmpl_s, int(duration_exmpl_s*fs_Hz), endpoint=False)
plt.plot(t_exmpl, s_exmpl, linewidth=0.5)
plt.xlim((0,0.03))
plt.show()

ipd.Audio(s_exmpl, rate=fs_Hz)

## Benchmark Vectorized Functions 
note: full benchmark with comparison to non-vectorized functions and 10 repeats takes up to 10 minutes.

In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Set BENCHMARK_COMPARE = True to compare vecorized and reference impl. │
# │  Set BENCHMARK_COMPARE = False to measure vectorized impl. only        │
# └────────────────────────────────────────────────────────────────────────┘

BENCHMARK_COMPARE = False

N_bench = 512
repeats = 10
coefs_bench = coefs_saw(N_bench)

def _timeit(fn, *args, repeats=10):
  fn(*args) # warmup
  t0 = time.perf_counter()
  for _ in range(repeats):
    fn(*args)
  t1 = time.perf_counter()
  return (t1 - t0) / repeats

print(f"N_bench={N_bench}, repeats={repeats}\n")

# ------------------- uncomment one or more benchmark below ------------------ #

### prerotate
for tau in [0.0, 0.25, -0.5, 1.3, 11/16]:
  t_vec = _timeit(prerotate_vectorized, coefs_bench, tau, repeats=repeats)
  if BENCHMARK_COMPARE:
    t_ref = _timeit(prerotate, coefs_bench, tau, repeats=repeats)
    print(f"prerotate tau={tau:>5}: \t\tref {t_ref*1e3:8.3f} ms | vec {t_vec*1e3:8.3f} ms | x{t_ref/t_vec:6.2f}")
  else:
    print(f"prerotate tau={tau:>5}: \t\tvec {t_vec*1e3:8.3f} ms")
print("")


### transform_hard
for P in [0.5, 1.0, 1.5, 11/8, 2.345, 4.567, 8.912]:
  t_vec = _timeit(transform_hard_vectorized, coefs_bench, P, repeats=repeats)
  if BENCHMARK_COMPARE:
    t_ref = _timeit(transform_hard, coefs_bench, P, repeats=repeats)
    print(f"transform_hard P={P:>5}: \tref {t_ref*1e3:8.3f} ms | vec {t_vec*1e3:8.3f} ms | x{t_ref/t_vec:6.2f}")
  else:
    print(f"transform_hard P={P:>5}: \tvec {t_vec*1e3:8.3f} ms")
print("")


### transform_mirrored
for P in [0.5, 1.0, 1.5, 11/8, 2.345, 4.567, 8.912]:
  t_vec = _timeit(transform_mirrored_vectorized, coefs_bench, P, repeats=repeats)
  if BENCHMARK_COMPARE:
    t_ref = _timeit(transform_mirrored, coefs_bench, P, repeats=repeats)
    print(f"transform_mirrored P={P:>5}: \tref {t_ref*1e3:8.3f} ms | vec {t_vec*1e3:8.3f} ms | x{t_ref/t_vec:6.2f}")
  else:
    print(f"transform_mirrored P={P:>5}: \tvec {t_vec*1e3:8.3f} ms")
print("")


### transform_pulsar
for P in [0.5, 1.0, 1.5, 11/8, 2.345, 4.567, 8.912]:
  t_vec = _timeit(transform_pulsar_vectorized, coefs_bench, P, repeats=repeats)
  if BENCHMARK_COMPARE:
    t_ref = _timeit(transform_pulsar, coefs_bench, P, repeats=repeats)
    print(f"transform_pulsar P={P:>5}: \tref {t_ref*1e3:8.3f} ms | vec {t_vec*1e3:8.3f} ms | x{t_ref/t_vec:6.2f}")
  else:
    print(f"transform_pulsar P={P:>5}: \tvec {t_vec*1e3:8.3f} ms")
print("")

# ### FourierOscillator.generate
add_synth_bench_ref = FourierOscillator(fs_Hz, N_MAX)
add_synth_bench_ref.set_coefs(coefs_bench)
add_synth_bench_vec = FourierOscillator_vectorized(fs_Hz, N_MAX)
add_synth_bench_vec.set_coefs(coefs_bench)

n_smpl_bench = 5
for f_bench_Hz in [20, 94, 1000, 12345]:
  add_synth_bench_vec.set_f_fund(f_bench_Hz)
  t_vec = _timeit(add_synth_bench_vec.generate, n_smpl_bench, repeats=repeats)
  if BENCHMARK_COMPARE:
    add_synth_bench_ref.set_f_fund(f_bench_Hz)
    t_ref = _timeit(add_synth_bench_ref.generate, n_smpl_bench, repeats=repeats)
    print(f"FourierOscillator.generate f_fund={f_bench_Hz:>5}, n_smpl={n_smpl_bench:>5}: \tref {t_ref*1e3:8.3f} ms | vec {t_vec*1e3:8.3f} ms | x{t_ref/t_vec:6.2f}")
  else:
    print(f"FourierOscillator.generate f_fund={f_bench_Hz:>5}, n_smpl={n_smpl_bench:>5}: \tvec {t_vec*1e3:8.3f} ms")

n_smpl_bench = fs_Hz
for f_bench_Hz in [20, 94, 1000, 12345]:
  add_synth_bench_vec.set_f_fund(f_bench_Hz)
  t_vec = _timeit(add_synth_bench_vec.generate, n_smpl_bench, repeats=repeats)
  if BENCHMARK_COMPARE:
    add_synth_bench_ref.set_f_fund(f_bench_Hz)
    t_ref = _timeit(add_synth_bench_ref.generate, n_smpl_bench, repeats=repeats)
    print(f"FourierOscillator.generate f_fund={f_bench_Hz:>5}, n_smpl={n_smpl_bench:>5}: \tref {t_ref*1e3:8.3f} ms | vec {t_vec*1e3:8.3f} ms | x{t_ref/t_vec:6.2f}")
  else:
      print(f"FourierOscillator.generate f_fund={f_bench_Hz:>5}, n_smpl={n_smpl_bench:>5}: \tvec {t_vec*1e3:8.3f} ms")
